# Vision Assist V1 — Clean, Reproducible Workflow

**Purpose:** prepare the ROD dataset, train one proper YOLOv8n baseline, evaluate on validation data, inspect predictions, and save a compact experiment summary.

## Run order

Run cells **1 → 2 → 3 → 4 → 5 → 6** in order. Do not rerun random old cells from the previous notebook.

- Cell 1: environment and GPU check
- Cell 2: prepare dataset and convert polygon labels to bounding boxes
- Cell 3: train the 50-epoch baseline (best checkpoint selected using validation data)
- Cell 4: evaluate on validation data only
- Cell 5: visual inspection on validation examples
- Cell 6: save experiment metadata and metrics locally

**Research hygiene:** the existing baseline has already been evaluated on the test split. Do not use test metrics to tune this new training run. Once model choices are fixed using validation results, use the test split for one final, documented comparison.

This is an experimental prototype, not a safety-grade navigation system.

## 1. Environment and GPU

Enable **GPU** in Kaggle notebook settings before running.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys
import json
import random
import shutil
import time

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not enabled. In Kaggle, open Notebook Settings > Accelerator > GPU, "
        "then rerun this cell."
    )

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "ultralytics", "PyYAML"
    ])

from ultralytics import YOLO
import yaml

print("Ultralytics and YAML ready.")

## 2. Prepare the ROD dataset

This uses the ROD dataset attached to this Kaggle notebook. It creates a prepared dataset under `/kaggle/working/rod_detection` and converts polygon annotations to enclosing bounding boxes.

The source dataset is left unchanged. Images are symlinked to avoid copying them.

In [ ]:
SOURCE = Path(
    "/kaggle/input/datasets/abtinzandi/"
    "obstacle-detection-dataset/ROD-Dataset/dataset"
)
DATA_DIR = Path("/kaggle/working/rod_detection")
DATA_YAML = Path("/kaggle/working/rod_detection.yaml")
RUNS_DIR = Path("/kaggle/working/vision_assist")

if not SOURCE.exists():
    raise FileNotFoundError(
        f"ROD dataset not found at {SOURCE}. Attach the ROD dataset in Kaggle's Input panel."
    )

CLASS_NAMES = [
    "Bike", "Building", "Car", "Person", "Stairs", "Traffic sign",
    "Electrical Pole", "Road", "Motorcycle", "Dustbin", "Dog", "Manhole",
    "Tree", "Guard rail", "Pedestrian crosswalk", "Truck", "Bus", "Bench",
    "Traffic Cone", "Fire hydrant", "Teraffic Barrel", "Plant Pot",
    "Electrical Box", "Chair", "Bicycle Rack"
]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLITS = ["train", "valid", "test"]

# Recreate only the derived working copy; do not alter SOURCE.
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

summary = {}

for split in SPLITS:
    src_images = SOURCE / split / "images"
    src_labels = SOURCE / split / "labels"
    out_images = DATA_DIR / split / "images"
    out_labels = DATA_DIR / split / "labels"
    out_images.mkdir(parents=True)
    out_labels.mkdir(parents=True)

    if not src_images.is_dir() or not src_labels.is_dir():
        raise FileNotFoundError(f"Missing source images or labels for split: {split}")

    image_files = sorted(
        p for p in src_images.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )
    if not image_files:
        raise RuntimeError(f"No images found for {split}")

    for image_path in image_files:
        (out_images / image_path.name).symlink_to(image_path)

    bbox_count = 0
    polygon_count = 0
    skipped = 0
    label_files = list(src_labels.glob("*.txt"))

    for label_path in label_files:
        output_lines = []
        for line in label_path.read_text().splitlines():
            parts = line.split()
            if not parts:
                continue
            try:
                class_id = int(parts[0])
                coords = [float(x) for x in parts[1:]]
            except (ValueError, OverflowError):
                skipped += 1
                continue

            if not (0 <= class_id < len(CLASS_NAMES)):
                skipped += 1
                continue

            if len(coords) == 4:
                x, y, w, h = coords
                kind = "bbox"
            elif len(coords) >= 6 and len(coords) % 2 == 0:
                xs, ys = coords[0::2], coords[1::2]
                xmin, xmax = min(xs), max(xs)
                ymin, ymax = min(ys), max(ys)
                x, y = (xmin + xmax) / 2, (ymin + ymax) / 2
                w, h = xmax - xmin, ymax - ymin
                kind = "polygon"
            else:
                skipped += 1
                continue

            if (
                not all(0 <= v <= 1 for v in (x, y, w, h))
                or w <= 0 or h <= 0
            ):
                skipped += 1
                continue

            output_lines.append(
                f"{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}"
            )
            if kind == "bbox":
                bbox_count += 1
            else:
                polygon_count += 1

        (out_labels / label_path.name).write_text(
            "\n".join(output_lines) + ("\n" if output_lines else "")
        )

    missing_labels = [
        p.name for p in image_files
        if not (out_labels / f"{p.stem}.txt").exists()
    ]
    if missing_labels:
        raise RuntimeError(
            f"{split}: {len(missing_labels)} images lack label files; "
            f"examples: {missing_labels[:5]}"
        )

    summary[split] = {
        "images": len(image_files),
        "label_files": len(label_files),
        "bbox_annotations_preserved": bbox_count,
        "polygon_annotations_converted": polygon_count,
        "skipped_annotations": skipped,
    }
    print(split.upper(), summary[split])

if sum(v["skipped_annotations"] for v in summary.values()) > 0:
    raise RuntimeError("Some annotations were skipped. Inspect counts before training.")

config = {
    "path": str(DATA_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False))

print("\nDataset config:", DATA_YAML)
print("Dataset preparation complete.")

## 3. Train one proper baseline

This is the **only training cell** in this clean workflow. It trains for up to 50 epochs and selects `best.pt` using validation performance. Early stopping may stop before 50 epochs if validation stops improving.

If Kaggle's runtime limit interrupts training, use the saved `last.pt` checkpoint to resume rather than starting a duplicate run.

In [ ]:
EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 8
DEVICE = 0
SEED = 42
RUN_NAME = "rod_yolov8n_50ep"

# Explicit pretrained YOLOv8 Nano baseline.
model = YOLO("yolov8n.pt")

train_start = time.time()
train_result = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=2,
    cache=False,
    amp=True,
    pretrained=True,
    seed=SEED,
    deterministic=True,
    patience=10,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=False,
    save=True,
    save_period=5,
    plots=True,
    val=True,
    verbose=True,
)

RUN_DIR = Path(train_result.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"
TRAIN_SECONDS = time.time() - train_start

print("\nTraining finished.")
print("Run directory:", RUN_DIR)
print("Best checkpoint exists:", BEST_WEIGHTS.exists())
print("Last checkpoint exists:", LAST_WEIGHTS.exists())
print(f"Training time: {TRAIN_SECONDS / 60:.1f} minutes")

if not BEST_WEIGHTS.exists():
    raise FileNotFoundError(
        f"Best checkpoint not found at {BEST_WEIGHTS}. Check training output."
    )

## 4. Evaluate the selected checkpoint on validation data

Use validation metrics for development decisions. The test split was already evaluated for the smoke-test baseline, so this cell deliberately uses `split="val"`.

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
val_metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=2,
    plots=True,
    project=str(RUNS_DIR),
    name=f"{RUN_NAME}_validation",
    exist_ok=False,
)

METRICS = {
    "model": "YOLOv8n",
    "run_name": RUN_NAME,
    "epochs_requested": EPOCHS,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "validation_images": summary["valid"]["images"],
    "validation_mAP50": float(val_metrics.box.map50),
    "validation_mAP50_95": float(val_metrics.box.map),
    "validation_precision": float(val_metrics.box.mp),
    "validation_recall": float(val_metrics.box.mr),
    "training_seconds": float(TRAIN_SECONDS),
    "best_checkpoint": str(BEST_WEIGHTS),
}

print("\nVALIDATION METRICS")
for key, value in METRICS.items():
    print(f"{key}: {value}")

## 5. Inspect example predictions

These are validation images for qualitative inspection. They are not a substitute for quantitative evaluation.

In [ ]:
import matplotlib.pyplot as plt

valid_images = sorted(
    p for p in (DATA_DIR / "valid" / "images").iterdir()
    if p.suffix.lower() in IMAGE_EXTS
)
random.seed(SEED)
sample_paths = random.sample(valid_images, min(6, len(valid_images)))

for image_path in sample_paths:
    prediction = best_model.predict(
        source=str(image_path),
        conf=0.25,
        imgsz=IMAGE_SIZE,
        device=DEVICE,
        verbose=False,
    )[0]

    print(f"\n{image_path.name}: {len(prediction.boxes)} detections")
    for box in prediction.boxes:
        class_id = int(box.cls[0].item())
        confidence = float(box.conf[0].item())
        print(f"  {prediction.names[class_id]}: {confidence:.2f}")

    annotated = prediction.plot()
    plt.figure(figsize=(10, 7))
    plt.imshow(annotated[:, :, ::-1])
    plt.title(image_path.name)
    plt.axis("off")
    plt.show()

## 6. Save experiment metadata and metrics

This writes a JSON record and copies Ultralytics' result CSV/plots into a compact artifact folder. Upload this folder and the new best checkpoint to GitHub after checking the results.

Do not upload dataset images, label caches, API tokens, or Kaggle secrets.

In [ ]:
ARTIFACT_DIR = RUNS_DIR / f"{RUN_NAME}_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

METRICS["dataset_preparation"] = summary
METRICS["dataset_yaml"] = str(DATA_YAML)
METRICS["checkpoint_size_mb"] = round(
    BEST_WEIGHTS.stat().st_size / (1024 * 1024), 3
)

(ARTIFACT_DIR / "metrics.json").write_text(
    json.dumps(METRICS, indent=2)
)

results_csv = RUN_DIR / "results.csv"
if results_csv.exists():
    shutil.copy2(results_csv, ARTIFACT_DIR / "training_results.csv")

for name in [
    "results.png", "confusion_matrix.png", "confusion_matrix_normalized.png",
    "PR_curve.png", "F1_curve.png", "P_curve.png", "R_curve.png",
    "labels.jpg", "labels_correlogram.jpg"
]:
    candidate = RUN_DIR / name
    if candidate.exists():
        shutil.copy2(candidate, ARTIFACT_DIR / name)

print("Artifacts saved to:", ARTIFACT_DIR)
print("Checkpoint:", BEST_WEIGHTS)
print("Metrics:", ARTIFACT_DIR / "metrics.json")
print("\nNext: upload BEST_WEIGHTS and ARTIFACT_DIR contents to GitHub.")
print("Do not run test-split evaluation until the final model is selected.")

In [ ]:

from pathlib import Path
import subprocess
import hashlib

repo = Path("/kaggle/working/vision_assist/github_upload")
local_model = Path(
    "/kaggle/working/vision_assist/rod_yolov8n_50ep/weights/best.pt"
)
github_model = repo / "models" / "rod_yolov8n_50ep_best.pt"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("Local checkpoint exists:", local_model.exists())
print("Repository checkpoint exists:", github_model.exists())

if local_model.exists() and github_model.exists():
    print("Local size (MB):", round(local_model.stat().st_size / 1e6, 2))
    print("Repository size (MB):", round(github_model.stat().st_size / 1e6, 2))
    print("Checkpoint files match:", sha256(local_model) == sha256(github_model))

print("\nLatest repository commit:")
subprocess.run(
    ["git", "log", "-1", "--format=%h %s"],
    cwd=repo,
    check=True
)

print("\nWorking tree status:")
subprocess.run(["git", "status", "--short"], cwd=repo, check=True)


In [ ]:

import subprocess
from pathlib import Path

repo = Path("/kaggle/working/vision_assist/github_upload")

def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=repo,
        capture_output=True,
        text=True,
        check=True
    ).stdout.strip()

# Refresh information from GitHub
git("fetch", "origin")

branch = git("branch", "--show-current")
print("Current branch:", branch)
print("\nLatest remote commit:")
print(git("log", "-1", "--oneline", f"origin/{branch}"))

print("\nCommits that contain the model:")
print(git(
    "log", "-5", "--oneline", f"origin/{branch}",
    "--", "models/rod_yolov8n_50ep_best.pt"
) or "No model-upload commit found.")

print("\nFiles tracked in the remote branch:")
for path in [
    "models/rod_yolov8n_50ep_best.pt",
    "results/rod_yolov8n_50ep/metrics.json",
]:
    result = subprocess.run(
        ["git", "cat-file", "-e", f"origin/{branch}:{path}"],
        cwd=repo, capture_output=True, text=True
    )
    print(f"{path}: {'PRESENT' if result.returncode == 0 else 'MISSING'}")


# VISION ASSIST — PHASE 2

In [ ]:

# Restore the runtime dependencies — no model training.

%pip install -q ultralytics==8.4.175

import ultralytics
import torch

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert ultralytics.__version__ == "8.4.175"

print("Environment setup complete. Do not retrain the model.")


In [ ]:

from pathlib import Path
import urllib.request
import hashlib

# Restore the trained model from GitHub
CHECKPOINT = Path(
    "/kaggle/working/vision_assist/rod_yolov8n_50ep/weights/best.pt"
)
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

MODEL_URL = (
    "https://raw.githubusercontent.com/"
    "SudhaMsr/VisionAssist-MVP/main/"
    "models/rod_yolov8n_50ep_best.pt"
)

if not CHECKPOINT.exists():
    print("Downloading the existing trained model...")
    urllib.request.urlretrieve(MODEL_URL, CHECKPOINT)

assert CHECKPOINT.exists(), "Model download failed."
assert CHECKPOINT.stat().st_size > 1_000_000, (
    "Downloaded file is unexpectedly small."
)

digest = hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()

print("Model restored successfully.")
print("Checkpoint:", CHECKPOINT)
print("Size (MB):", round(CHECKPOINT.stat().st_size / 1e6, 2))
print("SHA-256:", digest)


In [ ]:

from pathlib import Path

candidates = [
    Path(
        "/kaggle/input/datasets/abtinzandi/"
        "obstacle-detection-dataset/ROD-Dataset/"
        "dataset/valid/images"
    ),
    Path(
        "/kaggle/input/obstacle-detection-dataset/"
        "ROD-Dataset/dataset/valid/images"
    ),
    Path("/kaggle/working/rod_detection/valid/images"),
]

VALID_IMAGES = next(
    (p for p in candidates if p.is_dir()),
    None
)

if VALID_IMAGES is None:
    print("Validation images not found.")
    print("In Kaggle, click Add Input and attach the ROD dataset.")
else:
    image_files = [
        p for p in VALID_IMAGES.iterdir()
        if p.suffix.lower() in {
            ".jpg", ".jpeg", ".png", ".bmp", ".webp"
        }
    ]
    print("Validation folder:", VALID_IMAGES)
    print("Validation images:", len(image_files))
    print("READY for Phase 2.")


In [ ]:

# ============================================================
# VISION ASSIST — PHASE 2A
# Test path-aware object detection on validation images
# ============================================================

from pathlib import Path
from ultralytics import YOLO
from PIL import Image, ImageDraw
import pandas as pd
import matplotlib.pyplot as plt
import torch
import json
import time

CHECKPOINT = Path(
    "/kaggle/working/vision_assist/rod_yolov8n_50ep/weights/best.pt"
)

OUTPUT_DIR = Path("/kaggle/working/vision_assist/phase2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert CHECKPOINT.exists(), "Trained checkpoint not found."
assert VALID_IMAGES.exists(), "Validation image folder not found."

# Keep the trained model frozen: no training or fine-tuning.
model = YOLO(str(CHECKPOINT))
device = 0 if torch.cuda.is_available() else "cpu"

# Experimental image-space path region.
# These coordinates are normalized to image width/height.
PATH_POLYGON = [
    (0.42, 0.55),
    (0.58, 0.55),
    (0.78, 1.00),
    (0.22, 1.00),
]

CONF_THRESHOLD = 0.25
IMAGE_SIZE = 640
MAX_IMAGES = 200  # Initial pilot; we can scale after checking results.

def inside_polygon(point, polygon):
    x, y = point
    inside = False
    j = len(polygon) - 1

    for i in range(len(polygon)):
        xi, yi = polygon[i]
        xj, yj = polygon[j]

        if ((yi > y) != (yj > y)):
            crossing_x = (
                (xj - xi) * (y - yi) / ((yj - yi) + 1e-12) + xi
            )
            if x < crossing_x:
                inside = not inside
        j = i

    return inside

image_files = sorted([
    p for p in VALID_IMAGES.iterdir()
    if p.suffix.lower() in {
        ".jpg", ".jpeg", ".png", ".bmp", ".webp"
    }
])[:MAX_IMAGES]

records = []
start = time.time()

for idx, image_path in enumerate(image_files, start=1):
    result = model.predict(
        source=str(image_path),
        imgsz=IMAGE_SIZE,
        conf=CONF_THRESHOLD,
        device=device,
        verbose=False
    )[0]

    with Image.open(image_path) as im:
        width, height = im.size

    if result.boxes is None:
        continue

    boxes = result.boxes.xyxy.cpu().numpy()
    confidences = result.boxes.conf.cpu().numpy()
    class_ids = result.boxes.cls.cpu().numpy().astype(int)

    for box, confidence, class_id in zip(
        boxes, confidences, class_ids
    ):
        x1, y1, x2, y2 = map(float, box)

        # Approximate the object's ground-contact point.
        # This is only an image-space heuristic.
        px = ((x1 + x2) / 2) / width
        py = y2 / height

        relevant = inside_polygon(
            (px, py), PATH_POLYGON
        )

        records.append({
            "image": image_path.name,
            "class_name": model.names[int(class_id)],
            "confidence": float(confidence),
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "bottom_center_x_norm": px,
            "bottom_center_y_norm": py,
            "baseline_alert": True,
            "path_relevant": relevant,
            "path_aware_alert": relevant,
        })

    if idx % 50 == 0 or idx == len(image_files):
        print(f"Processed {idx}/{len(image_files)} images")

df = pd.DataFrame(records)
csv_path = OUTPUT_DIR / "phase2_pilot_detections.csv"
df.to_csv(csv_path, index=False)

baseline_count = len(df)
path_count = int(df["path_aware_alert"].sum()) if not df.empty else 0
filtered = baseline_count - path_count

print("\n========== PHASE 2A PILOT ==========")
print("Images processed:", len(image_files))
print("Detections:", baseline_count)
print("Path-region detections:", path_count)
print("Detections filtered by region:", filtered)

if baseline_count:
    print(
        "Detection-count reduction (%):",
        round(100 * filtered / baseline_count, 2)
    )

print("Elapsed seconds:", round(time.time() - start, 2))
print("CSV saved:", csv_path)

# Display class-level counts.
if not df.empty:
    summary = (
        df.groupby("class_name")
        .agg(
            detections=("image", "size"),
            path_relevant=("path_aware_alert", "sum")
        )
        .sort_values("detections", ascending=False)
    )
    display(summary)

# Save experiment configuration and limitations.
metadata = {
    "phase": "2A pilot",
    "checkpoint": str(CHECKPOINT),
    "validation_images_processed": len(image_files),
    "confidence_threshold": CONF_THRESHOLD,
    "image_size": IMAGE_SIZE,
    "path_polygon_normalized": PATH_POLYGON,
    "interpretation": (
        "Exploratory image-space filtering only. "
        "Not a measure of false-alert reduction, hazard recall, "
        "real-world distance, or collision risk."
    )
}

(OUTPUT_DIR / "phase2_pilot_metadata.json").write_text(
    json.dumps(metadata, indent=2)
)

print("\nPilot complete. No model weights were changed.")


In [ ]:

from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display
import pandas as pd
import json

OUTPUT_DIR = Path("/kaggle/working/vision_assist/phase2")
CSV_PATH = OUTPUT_DIR / "phase2_pilot_detections.csv"

df = pd.read_csv(CSV_PATH)

# Recover original validation image paths.
VALID_IMAGES = Path(
    "/kaggle/input/datasets/abtinzandi/"
    "obstacle-detection-dataset/ROD-Dataset/"
    "dataset/valid/images"
)

PATH_POLYGON = [
    (0.42, 0.55),
    (0.58, 0.55),
    (0.78, 1.00),
    (0.22, 1.00),
]

# Inspect a varied sample rather than only the first six records.
sample_names = (
    df.groupby("image")
      .size()
      .sort_values(ascending=False)
      .head(6)
      .index.tolist()
)

for image_name in sample_names:
    image_path = VALID_IMAGES / image_name

    if not image_path.exists():
        print("Image not found:", image_path)
        continue

    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    draw = ImageDraw.Draw(image)

    polygon_px = [
        (int(x * width), int(y * height))
        for x, y in PATH_POLYGON
    ]
    draw.polygon(polygon_px, outline="yellow", width=4)

    rows = df[df["image"] == image_name]

    for _, row in rows.iterrows():
        box = [
            int(row["x1"]), int(row["y1"]),
            int(row["x2"]), int(row["y2"])
        ]

        retained = bool(row["path_aware_alert"])
        colour = "lime" if retained else "red"

        draw.rectangle(box, outline=colour, width=3)
        label = (
            f'{row["class_name"]} '
            f'{row["confidence"]:.2f}'
        )
        draw.text((box[0], max(0, box[1] - 14)),
                  label, fill=colour)

    print(
        f"{image_name}: {len(rows)} detections, "
        f"{int(rows['path_aware_alert'].sum())} retained"
    )
    display(image)

print("Yellow = experimental path region")
print("Green = retained; red = filtered out")


In [ ]:

from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("/kaggle/working/vision_assist/phase2")
detections_path = OUTPUT_DIR / "phase2_pilot_detections.csv"

df = pd.read_csv(detections_path)

# Create a manual-review template.
review = df[[
    "image",
    "class_name",
    "confidence",
    "x1", "y1", "x2", "y2",
    "path_relevant",
    "path_aware_alert",
]].copy()

review["human_path_relevance"] = ""
review["human_hazard_relevance"] = ""
review["review_notes"] = ""

review_path = OUTPUT_DIR / "phase2_human_review.csv"
review.to_csv(review_path, index=False)

print("Review template created:", review_path)
print("Detections to review:", len(review))
print("\nFor each row, fill in:")
print("human_path_relevance: yes / no / uncertain")
print("human_hazard_relevance: yes / no / uncertain")
print("review_notes: optional explanation")
print("\nDo not treat model confidence as proof of hazard relevance.")


In [ ]:

from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("/kaggle/working/vision_assist/phase2")

review_path = OUTPUT_DIR / "phase2_human_review.csv"
sample_path = OUTPUT_DIR / "phase2_human_review_sample30.csv"

review = pd.read_csv(review_path)

# Sample from both groups so we inspect retained and filtered detections.
retained = review[review["path_aware_alert"] == True]
filtered = review[review["path_aware_alert"] == False]

n_each = min(15, len(retained), len(filtered))

sample = pd.concat([
    retained.sample(n=n_each, random_state=42),
    filtered.sample(n=n_each, random_state=42)
]).sample(frac=1, random_state=42)

sample["human_path_relevance"] = ""
sample["human_hazard_relevance"] = ""
sample["review_notes"] = ""

sample.to_csv(sample_path, index=False)

print("Sample created:", sample_path)
print("Rows to review:", len(sample))
print("\nPolicy decisions:")
print(sample["path_aware_alert"].value_counts())
display(sample[[
    "image", "class_name", "confidence",
    "path_aware_alert", "human_path_relevance",
    "human_hazard_relevance", "review_notes"
]].head(10))


In [ ]:

from pathlib import Path
import pandas as pd
from PIL import Image, ImageDraw
import math

# Input and output
review_path = Path("/kaggle/working/vision_assist/phase2/phase2_human_review_sample30.csv")
dataset_root = Path("/kaggle/input/datasets/abtinzandi/obstacle-detection-dataset/ROD-Dataset/dataset")
output_path = Path("/kaggle/working/vision_assist/phase2/phase2_review_contact_sheet.jpg")

review = pd.read_csv(review_path)

# Find the original validation images
image_folders = [
    dataset_root / "valid" / "images",
    dataset_root / "test" / "images",
    dataset_root / "train" / "images",
]
image_lookup = {}
for folder in image_folders:
    if folder.exists():
        for p in folder.rglob("*"):
            if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                image_lookup.setdefault(p.name, p)

# Build one labelled thumbnail per review row
tiles = []
tile_w, tile_h = 320, 250

for idx, row in review.iterrows():
    path = image_lookup.get(str(row["image"]))

    tile = Image.new("RGB", (tile_w, tile_h), "white")
    draw = ImageDraw.Draw(tile)

    if path is None:
        draw.text((8, 8), f"Image not found: {row['image']}", fill="red")
    else:
        img = Image.open(path).convert("RGB")
        # Draw the detection box on the original image
        d = ImageDraw.Draw(img)
        box = tuple(int(row[k]) for k in ["x1", "y1", "x2", "y2"])
        d.rectangle(box, outline="red", width=max(2, img.width // 250))
        img.thumbnail((tile_w - 8, tile_h - 55))
        tile.paste(img, ((tile_w - img.width) // 2, 4))

        decision = "KEPT" if bool(row["path_aware_alert"]) else "FILTERED"
        label = f"Row {idx+1}: {row['class_name']} {row['confidence']:.2f}"
        decision_label = f"{decision} | {row['image']}"
        draw.text((6, tile_h - 45), label, fill="black")
        draw.text((6, tile_h - 25), decision_label, fill="blue" if decision == "KEPT" else "darkred")

    tiles.append(tile)

cols = 3
rows = math.ceil(len(tiles) / cols)
sheet = Image.new("RGB", (cols * tile_w, rows * tile_h), "#dddddd")

for i, tile in enumerate(tiles):
    sheet.paste(tile, ((i % cols) * tile_w, (i // cols) * tile_h))

sheet.save(output_path, quality=90)
print("Contact sheet saved:", output_path)
display(sheet)


In [ ]:

from pathlib import Path
import pandas as pd
from PIL import Image, ImageDraw
import zipfile

base = Path("/kaggle/input/datasets/abtinzandi/obstacle-detection-dataset/ROD-Dataset/dataset")
out_dir = Path("/kaggle/working/vision_assist/phase2/full_resolution_review")
out_dir.mkdir(parents=True, exist_ok=True)

review = pd.read_csv(
    "/kaggle/working/vision_assist/phase2/phase2_human_review_sample30.csv"
)

# Locate the original images
lookup = {}
for split in ["valid", "test", "train"]:
    folder = base / split / "images"
    if folder.exists():
        for p in folder.rglob("*"):
            if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                lookup.setdefault(p.name, p)

for i, row in review.iterrows():
    src = lookup.get(str(row["image"]))
    if src is None:
        print("Missing image:", row["image"])
        continue

    img = Image.open(src).convert("RGB")
    draw = ImageDraw.Draw(img)
    box = tuple(int(row[k]) for k in ["x1", "y1", "x2", "y2"])
    draw.rectangle(box, outline="red", width=max(3, img.width // 300))

    decision = "KEPT" if bool(row["path_aware_alert"]) else "FILTERED"
    label = (
        f"Review {i+1} | {row['class_name']} "
        f"| confidence {row['confidence']:.2f} | {decision}"
    )
    draw.rectangle((0, 0, img.width, min(42, img.height)), fill="white")
    draw.text((8, 8), label, fill="black")

    img.save(out_dir / f"review_{i+1:02d}_{src.stem}.jpg", quality=95)

zip_path = Path("/kaggle/working/vision_assist/phase2/full_resolution_review.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(out_dir.glob("*.jpg")):
        z.write(p, arcname=p.name)

print("Review archive:", zip_path)
print("Images included:", len(list(out_dir.glob("*.jpg"))))


In [ ]:

from pathlib import Path
import pandas as pd

# Reuse the existing Phase 2 pilot results
PHASE2_DIR = Path("/kaggle/working/vision_assist/phase2")
INPUT_CSV = PHASE2_DIR / "phase2_pilot_detections.csv"

if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"Could not find {INPUT_CSV}. "
        "Check whether the Phase 2 CSV still exists after the Kaggle restart."
    )

detections = pd.read_csv(INPUT_CSV)

print("Loaded existing detections:", len(detections))
print("\nAvailable columns:")
print(detections.columns.tolist())

print("\nFirst five rows:")
display(detections.head())

print("\nUnique images:", detections["image"].nunique())


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PHASE2_DIR = Path("/kaggle/working/vision_assist/phase2")
df = pd.read_csv(PHASE2_DIR / "phase2_pilot_detections.csv")

# Same image-space path polygon used in Phase 2.
# Coordinates are normalised to [0, 1].
polygon = np.array([
    [0.42, 0.55],
    [0.58, 0.55],
    [0.78, 1.00],
    [0.22, 1.00],
])

# Bounding-box coordinates are in pixels. Infer image dimensions from
# the original validation images so normalisation is correct.
from PIL import Image

dataset_root = Path(
    "/kaggle/input/datasets/abtinzandi/obstacle-detection-dataset/ROD-Dataset/dataset"
)
image_lookup = {}
for p in (dataset_root / "valid" / "images").rglob("*"):
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        image_lookup.setdefault(p.name, p)

image_sizes = {}
for name in df["image"].unique():
    path = image_lookup.get(name)
    if path is None:
        raise FileNotFoundError(f"Cannot find validation image: {name}")
    with Image.open(path) as im:
        image_sizes[name] = im.size  # width, height

# Compute the fraction of each bounding box inside the polygon.
# Use OpenCV if installed; otherwise stop instead of silently approximating.
import cv2

def polygon_overlap_fraction(row):
    width, height = image_sizes[row["image"]]
    box = np.array([
        [row["x1"] / width, row["y1"] / height],
        [row["x2"] / width, row["y1"] / height],
        [row["x2"] / width, row["y2"] / height],
        [row["x1"] / width, row["y2"] / height],
    ], dtype=np.float32)

    box_area = max(
        (box[1, 0] - box[0, 0]) * (box[2, 1] - box[1, 1]), 1e-9
    )
    intersection_area, _ = cv2.intersectConvexConvex(
        box, polygon.astype(np.float32)
    )
    return max(0.0, float(intersection_area)) / box_area

df["bbox_path_overlap"] = df.apply(polygon_overlap_fraction, axis=1)

# Policies
df["policy_baseline"] = df["baseline_alert"].astype(bool)
df["policy_current"] = df["path_aware_alert"].astype(bool)

# Candidate: announce if at least 20% of the box overlaps the path,
# OR the box's bottom-centre is inside the path.
# This is a candidate for testing, not a claim of improved safety.
df["policy_overlap_candidate"] = (
    (df["bbox_path_overlap"] >= 0.20)
    | df["path_relevant"].astype(bool)
)

summary = pd.DataFrame([
    {
        "policy": "Baseline",
        "detections_alerted": int(df["policy_baseline"].sum()),
        "images_with_alerts": int(
            df.groupby("image")["policy_baseline"].any().sum()
        ),
    },
    {
        "policy": "Current bottom-centre",
        "detections_alerted": int(df["policy_current"].sum()),
        "images_with_alerts": int(
            df.groupby("image")["policy_current"].any().sum()
        ),
    },
    {
        "policy": "Bounding-box overlap candidate",
        "detections_alerted": int(df["policy_overlap_candidate"].sum()),
        "images_with_alerts": int(
            df.groupby("image")["policy_overlap_candidate"].any().sum()
        ),
    },
])

summary["detections_suppressed_vs_baseline"] = (
    int(df["policy_baseline"].sum()) - summary["detections_alerted"]
)
summary["suppression_percent"] = (
    100 * summary["detections_suppressed_vs_baseline"]
    / max(int(df["policy_baseline"].sum()), 1)
).round(2)

display(summary)

# Save reproducible per-detection decisions and summary.
df.to_csv(PHASE2_DIR / "phase2_policy_comparison.csv", index=False)
summary.to_csv(PHASE2_DIR / "phase2_policy_summary.csv", index=False)

print("Saved:", PHASE2_DIR / "phase2_policy_comparison.csv")
print("Saved:", PHASE2_DIR / "phase2_policy_summary.csv")


In [ ]:

from pathlib import Path
import json
import pandas as pd
from datetime import datetime, timezone

out = Path("/kaggle/working/vision_assist/phase2")
out.mkdir(parents=True, exist_ok=True)

# Reuse the existing comparison; load it if the notebook state changed.
comparison_path = out / "phase2_policy_comparison.csv"
summary_path = out / "phase2_policy_summary.csv"

if "df" not in globals() or "policy_overlap_candidate" not in df.columns:
    df = pd.read_csv(comparison_path)

summary = pd.read_csv(summary_path)

# 1. Save detections where the current and candidate policies disagree.
disagreements = df[
    df["policy_current"].astype(bool)
    != df["policy_overlap_candidate"].astype(bool)
].copy()
disagreements["decision_change"] = disagreements.apply(
    lambda r: (
        "candidate_adds_alert"
        if bool(r["policy_overlap_candidate"])
        else "candidate_removes_alert"
    ),
    axis=1,
)
disagreements.to_csv(out / "phase2_policy_disagreements.csv", index=False)

# 2. Per-class alert counts under each policy.
class_rows = []
for class_name, group in df.groupby("class_name"):
    class_rows.append({
        "class_name": class_name,
        "detections": len(group),
        "baseline_alerts": int(group["policy_baseline"].astype(bool).sum()),
        "current_alerts": int(group["policy_current"].astype(bool).sum()),
        "overlap_candidate_alerts": int(
            group["policy_overlap_candidate"].astype(bool).sum()
        ),
    })
pd.DataFrame(class_rows).sort_values(
    "detections", ascending=False
).to_csv(out / "phase2_policy_class_summary.csv", index=False)

# 3. Write a transparent report.
report = f"""VISION ASSIST V1 — PHASE 2 PILOT REPORT

Generated: {datetime.now(timezone.utc).isoformat()}

Purpose
Compare a baseline alert policy, the existing bottom-centre path filter,
and a bounding-box-overlap candidate on the same recorded detections.

Pilot
- Recorded detections: {len(df)}
- Images represented in detections: {df['image'].nunique()}
- Baseline alerts: {int(df['policy_baseline'].astype(bool).sum())}
- Current-policy alerts: {int(df['policy_current'].astype(bool).sum())}
- Overlap-candidate alerts: {int(df['policy_overlap_candidate'].astype(bool).sum())}

Observed alert-count reductions relative to baseline
- Current policy: 34.52%
- Overlap candidate: 32.38%

Interpretation
These are alert-count comparisons only. They do not demonstrate improved
hazard recall, fewer false alerts, collision avoidance, or user benefit.
The overlap candidate retained nine more detections than the current
policy in this pilot. The relevance of those additional detections has
not yet been independently established.

Limitations
- The analysis uses existing detections; the model was not retrained.
- The pilot was drawn from a particular dataset and may not represent
  real-world navigation scenes.
- Detection-level records alone cannot measure hazards missed by the
  detector.
- The path polygon is a fixed image-space heuristic, not a calibrated
  estimate of collision risk.
- The overlap candidate is experimental, not a validated safety policy.
- Human labels so far are preliminary and require verification against
  original images.

Next research requirement
Annotate relevant objects and missed hazards at image level, including
images with no detections. Then evaluate relevant-warning recall,
irrelevant-alert suppression, missed-hazard rate, and alert frequency
on a held-out evaluation set before making safety claims.
"""
(out / "PHASE2_PILOT_REPORT.txt").write_text(report, encoding="utf-8")

metadata = {
    "phase": "Phase 2 policy comparison",
    "model_retrained": False,
    "detections": int(len(df)),
    "images_with_detections": int(df["image"].nunique()),
    "policy_disagreements": int(len(disagreements)),
    "artifacts": [
        "phase2_policy_comparison.csv",
        "phase2_policy_summary.csv",
        "phase2_policy_disagreements.csv",
        "phase2_policy_class_summary.csv",
        "PHASE2_PILOT_REPORT.txt",
    ],
}
(out / "phase2_final_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

print("PHASE 2 ARTIFACTS SAVED")
for name in metadata["artifacts"] + ["phase2_final_metadata.json"]:
    print("-", out / name)
print("\nPolicy disagreements to inspect:", len(disagreements))
print("\nPreview of disagreements:")
display(disagreements[
    ["image", "class_name", "confidence", "policy_current",
     "policy_overlap_candidate", "bbox_path_overlap", "decision_change"]
].head(20))


In [ ]:

from pathlib import Path
import pandas as pd
from PIL import Image, ImageDraw
import math

out = Path("/kaggle/working/vision_assist/phase2")
df = pd.read_csv(out / "phase2_policy_disagreements.csv")

dataset = Path(
    "/kaggle/input/datasets/abtinzandi/obstacle-detection-dataset/ROD-Dataset/dataset/valid/images"
)

lookup = {}
for p in dataset.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        lookup.setdefault(p.name, p)

tiles = []
W, H = 480, 360

for _, row in df.iterrows():
    tile = Image.new("RGB", (W, H), "white")
    draw = ImageDraw.Draw(tile)
    path = lookup.get(str(row["image"]))

    if path is None:
        draw.text((10, 10), f"Image not found: {row['image']}", fill="red")
    else:
        img = Image.open(path).convert("RGB")
        box = tuple(int(row[k]) for k in ["x1", "y1", "x2", "y2"])
        ImageDraw.Draw(img).rectangle(box, outline="red", width=5)
        img.thumbnail((W - 10, H - 65))
        tile.paste(img, ((W - img.width) // 2, 5))
        draw.text((8, H - 55),
                  f"{row['image']} | {row['class_name']} "
                  f"| conf {row['confidence']:.2f}", fill="black")
        draw.text((8, H - 32),
                  f"Box overlap: {row['bbox_path_overlap']:.2f} | Candidate adds alert",
                  fill="darkblue")
    tiles.append(tile)

cols = 3
rows = math.ceil(len(tiles) / cols)
sheet = Image.new("RGB", (cols * W, rows * H), "#dddddd")
for i, tile in enumerate(tiles):
    sheet.paste(tile, ((i % cols) * W, (i // cols) * H))

path = out / "phase2_nine_disagreements.jpg"
sheet.save(path, quality=95)
print("Saved:", path)
display(sheet)


In [ ]:

from pathlib import Path
import pandas as pd
from PIL import Image, ImageDraw
import math

BASE = Path("/kaggle/input/datasets/abtinzandi/obstacle-detection-dataset/ROD-Dataset/dataset")
OUT = Path("/kaggle/working/vision_assist/phase2")
OUT.mkdir(parents=True, exist_ok=True)

detections = pd.read_csv(OUT / "phase2_pilot_detections.csv")

# Build a list of the 200 pilot images, including images with no detections.
image_names = sorted(detections["image"].dropna().unique())
image_dir = BASE / "valid" / "images"
lookup = {p.name: p for p in image_dir.rglob("*")
          if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}}

missing = [name for name in image_names if name not in lookup]
if missing:
    raise FileNotFoundError(f"Validation images missing: {missing[:10]}")

# Create one annotation row per image.
scene_file = OUT / "phase2_scene_suitability.csv"
if scene_file.exists():
    scenes = pd.read_csv(scene_file)
    scenes = scenes.drop_duplicates("image").set_index("image")
else:
    scenes = pd.DataFrame(index=image_names)

scenes = scenes.reindex(image_names)
scenes.index.name = "image"
scenes["scene_suitability"] = scenes.get(
    "scene_suitability", pd.Series(index=scenes.index, dtype="object")
).fillna("")
scenes["review_notes"] = scenes.get(
    "review_notes", pd.Series(index=scenes.index, dtype="object")
).fillna("")

# Count detections per image; zero-detection images will show as zero.
counts = detections.groupby("image").size()
scenes["detection_count"] = scenes.index.to_series().map(counts).fillna(0).astype(int)
scenes.reset_index().to_csv(scene_file, index=False)

# Generate a contact sheet in batches of 40 for manual scene annotation.
# Images are thumbnails only; inspect the original when a scene is ambiguous.
batch_dir = OUT / "scene_suitability_contact_sheets"
batch_dir.mkdir(exist_ok=True)
tile_w, tile_h = 240, 190
per_sheet = 40
cols = 4

for start in range(0, len(image_names), per_sheet):
    batch = image_names[start:start + per_sheet]
    rows = math.ceil(len(batch) / cols)
    sheet = Image.new("RGB", (cols * tile_w, rows * tile_h), "white")

    for j, name in enumerate(batch):
        img = Image.open(lookup[name]).convert("RGB")
        img.thumbnail((tile_w - 8, tile_h - 40))
        tile = Image.new("RGB", (tile_w, tile_h), "white")
        tile.paste(img, ((tile_w - img.width) // 2, 4))
        draw = ImageDraw.Draw(tile)
        count = int(counts.get(name, 0))
        draw.text((5, tile_h - 32), f"{start+j+1}. {name}", fill="black")
        draw.text((5, tile_h - 16), f"Detections: {count}", fill="black")
        sheet.paste(tile, ((j % cols) * tile_w, (j // cols) * tile_h))

    sheet.save(batch_dir / f"scene_review_{start+1:03d}_{start+len(batch):03d}.jpg",
               quality=90)

print("Scene annotation template:", scene_file)
print("Contact sheets:", batch_dir)
print("Pilot images:", len(image_names))
print("Images with zero recorded detections:",
      int((scenes["detection_count"] == 0).sum()))
print("Review labels: navigation_scene, non_navigation_scene, uncertain")
print("Open the contact sheets from the output files and label the CSV.")


In [ ]:
from pathlib import Path
import torch

MODEL_PATH = Path(
    "/kaggle/working/vision_assist/rod_yolov8n_50ep/weights/best.pt"
)

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("Checkpoint exists:", MODEL_PATH.exists())

if MODEL_PATH.exists():
    print("Checkpoint size (MB):", round(MODEL_PATH.stat().st_size / 1e6, 2))
else:
    print("Checkpoint missing — we'll restore it from GitHub.")

In [ ]:

from pathlib import Path
from IPython.display import display, Audio
from ultralytics import YOLO
import cv2
import numpy as np
import pandas as pd
import subprocess
import shutil
import importlib.util
import torch

MODEL_PATH = Path(
    "/kaggle/working/vision_assist/rod_yolov8n_50ep/weights/best.pt"
)

assert MODEL_PATH.exists(), f"Checkpoint not found: {MODEL_PATH}"

model = YOLO(str(MODEL_PATH))

print("Vision Assist model loaded successfully.")
print("Checkpoint:", MODEL_PATH.name)
print("GPU available:", torch.cuda.is_available())
print("Number of classes:", len(model.names))
print("\nAvailable speech/audio tools:")
print("ffmpeg:", shutil.which("ffmpeg"))
print("espeak:", shutil.which("espeak"))
print("espeak-ng:", shutil.which("espeak-ng"))
print("gTTS installed:", importlib.util.find_spec("gtts") is not None)
print("IPython Audio available: Yes")


In [ ]:

import subprocess
import sys

print("Installing offline speech support...")

result = subprocess.run(
    ["apt-get", "update", "-qq"],
    capture_output=True,
    text=True
)

result = subprocess.run(
    ["apt-get", "install", "-y", "-qq", "espeak-ng"],
    capture_output=True,
    text=True
)

import shutil

speech_engine = shutil.which("espeak-ng") or shutil.which("espeak")

if speech_engine:
    print("Speech engine ready:", speech_engine)
else:
    print("Speech engine installation was unsuccessful.")
    print(result.stderr[-1500:])


In [ ]:

from pathlib import Path
import subprocess
import shutil
import numpy as np
from IPython.display import display, Audio, Image as NotebookImage

# Reuse the already-loaded model
OUTPUT_DIR = Path("/kaggle/working/vision_assist/v1_demo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Find sample images from the validation dataset
DATASET_ROOT = Path(
    "/kaggle/input/datasets/abtinzandi/obstacle-detection-dataset/ROD-Dataset/dataset"
)

image_dirs = [
    DATASET_ROOT / "valid" / "images",
    DATASET_ROOT / "test" / "images",
    DATASET_ROOT / "train" / "images",
]

image_files = []
for folder in image_dirs:
    if folder.exists():
        image_files = sorted(
            p for p in folder.iterdir()
            if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
        )
        if image_files:
            break

assert image_files, "No dataset images found. Check DATASET_ROOT."

# Use a small batch for the first demo
sample_images = image_files[:8]
speech_engine = shutil.which("espeak-ng") or shutil.which("espeak")
assert speech_engine, "Speech engine is unavailable."

spoken_alerts = []
saved_frames = []

for i, image_path in enumerate(sample_images, start=1):
    result = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=0.35,
        verbose=False
    )[0]

    annotated = result.plot()
    frame_path = OUTPUT_DIR / f"frame_{i:02d}.jpg"

    import cv2
    cv2.imwrite(str(frame_path), annotated)
    saved_frames.append(frame_path)

    detected_names = []
    if result.boxes is not None:
        detected_names = sorted({
            model.names[int(cls)]
            for cls in result.boxes.cls.tolist()
        })

    if detected_names:
        # Keep alerts concise rather than speaking every box.
        alert = "Caution. " + ", ".join(detected_names[:3]) + " detected."
    else:
        alert = "No objects detected."

    spoken_alerts.append(alert)
    print(f"{i}/{len(sample_images)}: {image_path.name} -> {alert}")

# Create a spoken warning WAV file
speech_text = " ... ".join(spoken_alerts)
wav_path = OUTPUT_DIR / "vision_assist_warnings.wav"

subprocess.run(
    [speech_engine, "-v", "en", "-s", "145", "-w", str(wav_path), speech_text],
    check=True,
    capture_output=True,
    text=True
)

# Save the transcript and show the first annotated result
transcript_path = OUTPUT_DIR / "spoken_alerts.txt"
transcript_path.write_text("\n".join(spoken_alerts), encoding="utf-8")

print("\nDemo processing complete.")
print("Annotated images:", len(saved_frames))
print("Audio:", wav_path)
print("Transcript:", transcript_path)

display(NotebookImage(filename=str(saved_frames[0])))
display(Audio(filename=str(wav_path)))


In [ ]:

from pathlib import Path
import cv2
import subprocess
import shutil

OUTPUT_DIR = Path("/kaggle/working/vision_assist/v1_demo")
frames = sorted(OUTPUT_DIR.glob("frame_*.jpg"))

assert frames, "No annotated frames found. Run Step 4 first."

# Create a video from the annotated frames
first = cv2.imread(str(frames[0]))
height, width = first.shape[:2]

silent_video = OUTPUT_DIR / "vision_assist_video_silent.mp4"
final_video = OUTPUT_DIR / "vision_assist_v1_demo.mp4"
fps = 1  # One sample image per second

writer = cv2.VideoWriter(
    str(silent_video),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

for frame_path in frames:
    frame = cv2.imread(str(frame_path))
    if frame is None:
        continue
    if frame.shape[:2] != (height, width):
        frame = cv2.resize(frame, (width, height))
    writer.write(frame)

writer.release()

# Combine the video with the spoken warnings
ffmpeg = shutil.which("ffmpeg")
audio = OUTPUT_DIR / "vision_assist_warnings.wav"

assert ffmpeg, "ffmpeg is unavailable."
assert audio.exists(), "Speech audio file not found."

subprocess.run([
    ffmpeg, "-y",
    "-i", str(silent_video),
    "-i", str(audio),
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-shortest",
    str(final_video)
], check=True, capture_output=True, text=True)

print("Vision Assist V1 demo video created:")
print(final_video)
print("Video duration is approximately", len(frames), "seconds.")


In [ ]:

from IPython.display import Video, display
from pathlib import Path

video_path = Path(
    "/kaggle/working/vision_assist/v1_demo/vision_assist_v1_demo.mp4"
)

assert video_path.exists(), "Demo video not found."

print(f"Video size: {video_path.stat().st_size / (1024 * 1024):.2f} MB")
display(Video(str(video_path), embed=True, width=720))


In [ ]:

from pathlib import Path
import subprocess
import shutil
import cv2
import wave
import numpy as np
from IPython.display import Audio, display

OUT = Path("/kaggle/working/vision_assist/v1_demo")
frames = sorted(OUT.glob("frame_*.jpg"))
engine = shutil.which("espeak-ng") or shutil.which("espeak")

assert len(frames) == len(spoken_alerts) and len(frames) > 0
assert engine, "Speech engine not found."

# Generate one audio segment per image, preserving each detected class.
segments = []
for i, alert in enumerate(spoken_alerts):
    wav_path = OUT / f"alert_{i:02d}.wav"
    subprocess.run(
        [engine, "-v", "en", "-s", "145", "-w", str(wav_path), alert],
        check=True, capture_output=True
    )
    with wave.open(str(wav_path), "rb") as w:
        params = w.getparams()
        audio_data = w.readframes(w.getnframes())

    # Add silence after each warning so it is easier to hear.
    silence = b"\x00" * int(params.sampwidth * params.nchannels * params.framerate * 0.5)
    segments.append((params, audio_data + silence))

# Combine audio segments
combined = OUT / "vision_assist_warnings_v2.wav"
with wave.open(str(combined), "wb") as w:
    w.setparams(segments[0][0])
    for _, data in segments:
        w.writeframes(data)

# Show each annotated image for about 2 seconds.
silent_video = OUT / "vision_assist_video_v2_silent.mp4"
final_video = OUT / "vision_assist_v1_demo_v2.mp4"

first = cv2.imread(str(frames[0]))
h, width = first.shape[:2]
fps = 2
seconds_per_frame = 2
writer = cv2.VideoWriter(
    str(silent_video),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps, (width, h)
)

for path in frames:
    frame = cv2.imread(str(path))
    if frame.shape[:2] != (h, width):
        frame = cv2.resize(frame, (width, h))
    for _ in range(fps * seconds_per_frame):
        writer.write(frame)
writer.release()

subprocess.run([
    shutil.which("ffmpeg"), "-y",
    "-i", str(silent_video),
    "-i", str(combined),
    "-c:v", "libx264", "-pix_fmt", "yuv420p",
    "-c:a", "aac", "-shortest",
    str(final_video)
], check=True, capture_output=True, text=True)

print("Updated video:", final_video)
print("Warnings included:")
for alert in spoken_alerts:
    print("-", alert)

display(Audio(filename=str(combined)))


In [ ]:
from pathlib import Path

for folder in [Path("/kaggle/input"), Path("/kaggle/working")]:
    print(f"\n--- {folder} ---")
    if folder.exists():
        for item in list(folder.iterdir())[:30]:
            print(item)
    else:
        print("Folder does not exist")

print("\n--- Current notebook environment ---")
print("Kaggle input exists:", Path("/kaggle/input").exists())
print("Working directory exists:", Path("/kaggle/working").exists())

In [ ]:
from pathlib import Path

for root in [Path("/kaggle/input/notebooks"), Path("/kaggle/input/datasets")]:
    print(f"\n--- {root} ---")
    if root.exists():
        for p in list(root.rglob("*"))[:100]:
            print(("DIR  " if p.is_dir() else "FILE ") + str(p))
    else:
        print("Not found")

In [3]:
from pathlib import Path

root = Path("/kaggle/working")
print("Working directory:", list(root.iterdir()))

Working directory: [PosixPath('/kaggle/working/.virtual_documents')]


In [4]:

import os
import subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO_URL = "https://github.com/SudhaMsr/VisionAssist-MVP.git"
REPO_DIR = Path("/kaggle/working/VisionAssist-MVP")

# Read the token securely from Kaggle Secrets
token = UserSecretsClient().get_secret("GITHUB_TOKEN")

# Temporary authentication helper; token is not saved in Git config
askpass = Path("/tmp/github-askpass.sh")
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) echo "x-access-token" ;;\n'
    '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
    '  *) exit 1 ;;\n'
    'esac\n'
)
askpass.chmod(0o700)

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = str(askpass)
env["GIT_TERMINAL_PROMPT"] = "0"

def git(args):
    subprocess.run(
        ["git", *args],
        cwd=REPO_DIR if REPO_DIR.exists() else None,
        env=env,
        check=True
    )

# Clone the existing repository, or update it if already cloned
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        env=env,
        check=True
    )
else:
    git(["fetch", "origin"])

git(["config", "user.name", "SudhaMsr"])
git(["config", "user.email", "YOUR_GITHUB_EMAIL"])

print("GitHub authentication and repository setup completed.")
print("Repository:", REPO_DIR)
print("Current branch:", subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=REPO_DIR, text=True
).strip())

# Remove the temporary helper and token from this cell's environment
askpass.unlink(missing_ok=True)
env.pop("GITHUB_TOKEN", None)
env.pop("GIT_ASKPASS", None)
del token


Cloning into '/kaggle/working/VisionAssist-MVP'...


GitHub authentication and repository setup completed.
Repository: /kaggle/working/VisionAssist-MVP
Current branch: main


In [5]:
from pathlib import Path
import subprocess

repo = Path("/kaggle/working/VisionAssist-MVP")

print("=== Existing GitHub repository files ===")
subprocess.run([
    "git", "-C", str(repo), "ls-files"
], check=True)

print("\n=== Available project files in /kaggle/working ===")
for p in sorted(Path("/kaggle/working").iterdir()):
    if p.name != "VisionAssist-MVP":
        print(("DIR  " if p.is_dir() else "FILE ") + p.name)

print("\n=== Existing model checkpoint ===")
model = repo / "models/rod_yolov8n_50ep_best.pt"
print("Found in repository:", model.exists())

print("\n=== Git status ===")
subprocess.run(["git", "-C", str(repo), "status", "--short"], check=True)

=== Existing GitHub repository files ===
README.md
configs/rod_detection.yaml
models/rod_baseline_smoke_best.pt
models/rod_yolov8n_50ep_best.pt
notebooks/Vision_Assist_V2_Voice_Demo.ipynb
notebooks/vision-assist-v1.ipynb
results/rod_yolov8n_50ep/confusion_matrix.png
results/rod_yolov8n_50ep/confusion_matrix_normalized.png
results/rod_yolov8n_50ep/labels.jpg
results/rod_yolov8n_50ep/metrics.json
results/rod_yolov8n_50ep/results.png
results/rod_yolov8n_50ep/training_results.csv

=== Available project files in /kaggle/working ===
DIR  .virtual_documents

=== Existing model checkpoint ===
Found in repository: True

=== Git status ===


CompletedProcess(args=['git', '-C', '/kaggle/working/VisionAssist-MVP', 'status', '--short'], returncode=0)

In [6]:
import subprocess
from pathlib import Path

repo = Path("/kaggle/working/VisionAssist-MVP")

def git(*args):
    return subprocess.run(
        ["git", "-C", str(repo), *args],
        check=True, text=True, capture_output=True
    )

# Ensure we're up to date before pushing
git("pull", "--rebase", "origin", "main")

# Stage repository changes
git("add", "-A")
status = git("status", "--porcelain").stdout.strip()

if status:
    print("Changes to commit:")
    print(status)
    git("commit", "-m", "Update Vision Assist project files")
    git("push", "origin", "main")
    print("\nPush completed.")
else:
    print("No new changes to push — repository is already up to date.")

print("\nLatest commit:")
print(git("log", "-1", "--oneline").stdout)
print("\nRepository:", "https://github.com/SudhaMsr/VisionAssist-MVP")

No new changes to push — repository is already up to date.

Latest commit:
5da56cb Add files via upload


Repository: https://github.com/SudhaMsr/VisionAssist-MVP


In [8]:
import os
import subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

repo = Path("/kaggle/working/VisionAssist-MVP")
token = UserSecretsClient().get_secret("GITHUB_TOKEN")

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_TERMINAL_PROMPT"] = "0"

askpass = Path("/tmp/github-askpass.sh")
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) echo "x-access-token" ;;\n'
    '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
    '  *) exit 1 ;;\n'
    'esac\n'
)
askpass.chmod(0o700)
env["GIT_ASKPASS"] = str(askpass)

result = subprocess.run(
    ["git", "-C", str(repo), "push", "origin", "main"],
    env=env, capture_output=True, text=True
)

print("Exit code:", result.returncode)
print("Git output:", result.stdout)
print("Error:", result.stderr)

askpass.unlink(missing_ok=True)
del token

Exit code: 0
Git output: 
Error: To https://github.com/SudhaMsr/VisionAssist-MVP.git
   5da56cb..f72ebcf  main -> main



In [9]:
from pathlib import Path

for root in [Path("/kaggle/working/VisionAssist-MVP/notebooks")]:
    print("Notebook files:")
    for p in root.glob("*.ipynb"):
        print(p.name, f"({p.stat().st_size / 1024:.1f} KB)")

Notebook files:
vision-assist-v1.ipynb (3224.5 KB)
Vision_Assist_V2_Voice_Demo.ipynb (15.0 KB)


In [12]:

import os
import subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

repo = Path("/kaggle/working/VisionAssist-MVP")
token = UserSecretsClient().get_secret("GITHUB_TOKEN")

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_TERMINAL_PROMPT"] = "0"

askpass = Path("/tmp/github-askpass.sh")
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) echo "x-access-token" ;;\n'
    '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
    '  *) exit 1 ;;\n'
    'esac\n'
)
askpass.chmod(0o700)
env["GIT_ASKPASS"] = str(askpass)

def git(*args):
    return subprocess.run(
        ["git", "-C", str(repo), *args],
        env=env, check=True, capture_output=True, text=True
    )

try:
    # Sync before committing to avoid pushing over newer remote changes
    git("pull", "--rebase", "origin", "main")

    # Stage all current tracked and untracked repository files
    git("add", "-A")
    changes = git("status", "--porcelain").stdout.strip()

    if changes:
        print("Changes being committed:")
        print(changes)
        git("commit", "-m", "Prepare Vision Assist project for developer handoff")
    else:
        print("No file changes found; repository is already committed.")

    # Push current branch to GitHub
    git("push", "origin", "main")

    print("\nLatest commit:")
    print(git("log", "-1", "--oneline").stdout.strip())
    print("\nGitHub: https://github.com/SudhaMsr/VisionAssist-MVP")
    print("Push completed successfully.")

finally:
    askpass.unlink(missing_ok=True)
    env.pop("GITHUB_TOKEN", None)
    env.pop("GIT_ASKPASS", None)
    del token


No file changes found; repository is already committed.

Latest commit:
f72ebcf Force sync Vision Assist project to GitHub

GitHub: https://github.com/SudhaMsr/VisionAssist-MVP
Push completed successfully.


In [13]:
from pathlib import Path

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/temp"),
]

extensions = {".mp4", ".wav", ".mp3", ".png", ".jpg", ".json", ".csv", ".ipynb"}

print("=== Voice demo and result files found ===")
for root in roots:
    if root.exists():
        for p in root.rglob("*"):
            if p.is_file() and (
                p.suffix.lower() in extensions
                and any(k in str(p).lower() for k in [
                    "vision_assist", "v1_demo", "voice", "rod_yolov8n"
                ])
            ):
                print(f"{p.stat().st_size / 1024:.1f} KB  {p}")

=== Voice demo and result files found ===
15.0 KB  /kaggle/working/VisionAssist-MVP/notebooks/Vision_Assist_V2_Voice_Demo.ipynb
1.1 KB  /kaggle/working/VisionAssist-MVP/results/rod_yolov8n_50ep/metrics.json
8.1 KB  /kaggle/working/VisionAssist-MVP/results/rod_yolov8n_50ep/training_results.csv
187.5 KB  /kaggle/working/VisionAssist-MVP/results/rod_yolov8n_50ep/labels.jpg
528.0 KB  /kaggle/working/VisionAssist-MVP/results/rod_yolov8n_50ep/confusion_matrix_normalized.png
459.2 KB  /kaggle/working/VisionAssist-MVP/results/rod_yolov8n_50ep/confusion_matrix.png
265.6 KB  /kaggle/working/VisionAssist-MVP/results/rod_yolov8n_50ep/results.png


In [15]:
import os
import subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

repo = Path("/kaggle/working/VisionAssist-MVP")
token = UserSecretsClient().get_secret("GITHUB_TOKEN")

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_TERMINAL_PROMPT"] = "0"

askpass = Path("/tmp/github-askpass.sh")
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) echo "x-access-token" ;;\n'
    '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
    '  *) exit 1 ;;\n'
    'esac\n'
)
askpass.chmod(0o700)
env["GIT_ASKPASS"] = str(askpass)

def run_git(*args):
    result = subprocess.run(
        ["git", "-C", str(repo), *args],
        env=env, capture_output=True, text=True
    )
    print("$ git", " ".join(args))
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode:
        raise RuntimeError(f"Git failed with exit code {result.returncode}")
    return result

try:
    run_git("add", "DEVELOPER_HANDOFF.md")

    staged = run_git("diff", "--cached", "--quiet")
    # --quiet returns 1 when staged changes exist; handle separately below
except RuntimeError:
    pass

try:
    status = subprocess.run(
        ["git", "-C", str(repo), "status", "--short"],
        env=env, capture_output=True, text=True, check=True
    ).stdout
    if "DEVELOPER_HANDOFF.md" in status:
        run_git("commit", "-m", "Add Vision Assist developer handoff")

    run_git("push", "origin", "main")
    print("\nSUCCESS: https://github.com/SudhaMsr/VisionAssist-MVP")
finally:
    askpass.unlink(missing_ok=True)
    env.pop("GITHUB_TOKEN", None)
    env.pop("GIT_ASKPASS", None)
    del token

$ git add DEVELOPER_HANDOFF.md
$ git diff --cached --quiet
$ git push origin main
To https://github.com/SudhaMsr/VisionAssist-MVP.git
   f72ebcf..637591b  main -> main


SUCCESS: https://github.com/SudhaMsr/VisionAssist-MVP


In [16]:

import os, sys, glob, math, wave, shutil, subprocess
from pathlib import Path

# ---------- Paths ----------
repo = Path("/kaggle/working/VisionAssist-MVP")
model_path = repo / "models/rod_yolov8n_50ep_best.pt"
image_dir = Path(
    "/kaggle/input/datasets/abtinzandi/"
    "obstacle-detection-dataset/ROD-Dataset/dataset/valid/images"
)
out = repo / "results/voice_demo"
out.mkdir(parents=True, exist_ok=True)

assert model_path.is_file(), f"Existing checkpoint missing: {model_path}"
assert image_dir.is_dir(), f"Validation images not found: {image_dir}"

# ---------- Tools ----------
if shutil.which("espeak-ng") is None or shutil.which("ffmpeg") is None:
    subprocess.run(
        ["bash", "-lc",
         "apt-get update -qq && apt-get install -y -qq espeak-ng ffmpeg"],
        check=True
    )

try:
    from ultralytics import YOLO
    import cv2
    import numpy as np
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "ultralytics", "opencv-python-headless"],
        check=True
    )
    from ultralytics import YOLO
    import cv2
    import numpy as np

# ---------- Reuse trained model; do NOT train ----------
model = YOLO(str(model_path))
images = sorted(
    p for p in image_dir.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)[:8]
assert images, f"No images found in {image_dir}"

fps = 10
frame_size = None
video_frames = []
audio_segments = []
summary = []

for i, image_path in enumerate(images, 1):
    result = model.predict(
        source=str(image_path), conf=0.25, verbose=False
    )[0]

    annotated = result.plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    h, w = annotated.shape[:2]

    # Resize frames consistently for the video
    if frame_size is None:
        frame_size = (w, h)
    elif (w, h) != frame_size:
        annotated = cv2.resize(annotated, frame_size)

    if result.boxes is not None and len(result.boxes) > 0:
        best_idx = int(result.boxes.conf.argmax().item())
        cls_id = int(result.boxes.cls[best_idx].item())
        label = str(result.names[cls_id])
        confidence = float(result.boxes.conf[best_idx].item())
        spoken = f"{label} detected"
        summary.append(
            f"{image_path.name}: {label} ({confidence:.2f})"
        )
    else:
        spoken = "No objects detected"
        summary.append(f"{image_path.name}: no detections")

    # Save annotated image and synthesize spoken alert
    cv2.imwrite(
        str(out / f"frame_{i:02d}.jpg"),
        cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR)
    )
    wav_path = out / f"speech_{i:02d}.wav"
    subprocess.run(
        ["espeak-ng", "-v", "en", "-s", "145",
         "-w", str(wav_path), spoken],
        check=True, capture_output=True
    )

    with wave.open(str(wav_path), "rb") as wf:
        params = wf.getparams()
        audio = wf.readframes(wf.getnframes())
        duration = wf.getnframes() / wf.getframerate()

    if i == 1:
        audio_params = params
    else:
        assert (
            params.nchannels == audio_params.nchannels
            and params.sampwidth == audio_params.sampwidth
            and params.framerate == audio_params.framerate
        ), "Speech WAV settings do not match"

    silence_frames = int(params.framerate * 0.7)
    silence = b"\x00" * silence_frames * params.nchannels * params.sampwidth
    audio_segments.append(audio + silence)

    # Hold each annotated frame while its warning is spoken
    frame_count = max(1, math.ceil((duration + 0.7) * fps))
    video_frames.extend([annotated] * frame_count)

# Write silent video
silent_video = out / "vision_assist_demo_silent.mp4"
height, width = video_frames[0].shape[:2]
writer = cv2.VideoWriter(
    str(silent_video),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps, (width, height)
)
assert writer.isOpened(), "Could not initialize video writer"
for frame in video_frames:
    writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
writer.release()

# Join spoken alerts into one audio track
combined_wav = out / "vision_assist_voice.wav"
with wave.open(str(combined_wav), "wb") as wf:
    wf.setparams(audio_params)
    for segment in audio_segments:
        wf.writeframes(segment)

# Mux voice and video
final_mp4 = out / "vision_assist_voice_demo.mp4"
subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error",
    "-i", str(silent_video), "-i", str(combined_wav),
    "-c:v", "libx264", "-pix_fmt", "yuv420p",
    "-c:a", "aac", "-shortest", str(final_mp4)
], check=True)

(out / "detections.txt").write_text("\n".join(summary) + "\n")
silent_video.unlink(missing_ok=True)

print("Demo generated:")
for p in sorted(out.iterdir()):
    if p.is_file():
        print(f"{p.name}: {p.stat().st_size / 1024:.1f} KB")

# ---------- Push generated demo files to GitHub ----------
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_TERMINAL_PROMPT"] = "0"

askpass = Path("/tmp/github-askpass.sh")
askpass.write_text(
    '#!/bin/sh\ncase "$1" in\n'
    '  *Username*) echo "x-access-token" ;;\n'
    '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
    '  *) exit 1 ;;\nesac\n'
)
askpass.chmod(0o700)
env["GIT_ASKPASS"] = str(askpass)

def git(*args):
    return subprocess.run(
        ["git", "-C", str(repo), *args],
        env=env, capture_output=True, text=True, check=True
    )

try:
    git("pull", "--rebase", "origin", "main")
    git("add", "results/voice_demo")
    staged = subprocess.run(
        ["git", "-C", str(repo), "diff", "--cached", "--quiet"],
        capture_output=True
    )
    if staged.returncode == 1:
        git("commit", "-m", "Add generated Vision Assist voice demo")
    git("push", "origin", "main")
    print("\nSUCCESS: Voice demo pushed to GitHub.")
    print("Folder: results/voice_demo/")
    print("Repository: https://github.com/SudhaMsr/VisionAssist-MVP")
finally:
    askpass.unlink(missing_ok=True)
    env.pop("GITHUB_TOKEN", None)
    env.pop("GIT_ASKPASS", None)
    del token


W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


Selecting previously unselected package libpcaudio0:amd64.
(Reading database ... 129802 files and directories currently installed.)
Preparing to unpack .../libpcaudio0_1.2-2build3_amd64.deb ...
Unpacking libpcaudio0:amd64 (1.2-2build3) ...
Selecting previously unselected package libsonic0:amd64.
Preparing to unpack .../libsonic0_0.2.0-13build1_amd64.deb ...
Unpacking libsonic0:amd64 (0.2.0-13build1) ...
Selecting previously unselected package espeak-ng-data:amd64.
Preparing to unpack .../espeak-ng-data_1.51+dfsg-12build1_amd64.deb ...
Unpacking espeak-ng-data:amd64 (1.51+dfsg-12build1) ...
Selecting previously unselected package libespeak-ng1:amd64.
Preparing to unpack .../libespeak-ng1_1.51+dfsg-12build1_amd64.deb ...
Unpacking libespeak-ng1:amd64 (1.51+dfsg-12build1) ...
Selecting previously unselected package espeak-ng.
Preparing to unpack .../espeak-ng_1.51+dfsg-12build1_amd64.deb ...
Unpacking espeak-ng (1.51+dfsg-12build1) ...
Setting up libpcaudio0:amd64 (1.2-2build3) ...
Settin